# From Excel Metadata to a Bipartite Book-Place Network

This notebook demonstrates a complete data pipeline for teaching purposes:

1. Load British Library metadata from an Excel file into a Pandas DataFrame.
2. Extract place names from the `Imprint` field using spaCy (English model).
3. Query GeoNames (via `geopy`) to disambiguate places and mark ambiguous names.
4. Build a bipartite `networkx` graph with two node types: `book` and `place`.
5. Visualize the graph with `ipysigma`.
6. Export the network to `.gexf` for Gephi / Gephi Web.

---

## Teaching goals

- Show a reproducible metadata-to-network workflow.
- Keep complex logic in reusable functions.
- Make each processing step transparent and inspectable.

## 1) Setup and dependencies

This section imports required libraries and defines constants used throughout the notebook.

> If you are running this notebook in a fresh environment, install the dependencies first:
>
> ```bash
> pip install pandas spacy geopy networkx ipysigma openpyxl tqdm
> python -m spacy download en_core_web_sm
> ```

In [ ]:
! python -m spacy download en_core_web_sm

In [ ]:
# Import core libraries for tabular data, NLP, geocoding, and networks.
import os
import time
from functools import lru_cache
from typing import Dict, List

import pandas as pd
import networkx as nx
import spacy
from geopy.geocoders import GeoNames
from geopy.extra.rate_limiter import RateLimiter
from ipysigma import Sigma
from tqdm.auto import tqdm

# Enable tqdm progress bars for pandas operations (e.g., Series/DataFrame progress_apply).
tqdm.pandas()

# Define the source dataset URL and output path in one place.
EXCEL_URL = "https://github.com/mromanello/ADA-DHOxSS/raw/refs/heads/master/data/bl_books/extra_metadata.xls"
OUTPUT_GEXF_PATH = "../../data/books_places_bipartite.gexf"

# Try loading the spaCy English model.
# If it is not installed, raise a clear error message for students.
try:
    nlp = spacy.load("en_core_web_sm")
except OSError as exc:
    raise OSError(
        "spaCy model 'en_core_web_sm' is missing. Run: python -m spacy download en_core_web_sm"
    ) from exc

print("Setup complete.")

## 2) Load the Excel data

We load the spreadsheet into a DataFrame and inspect key columns, especially `Imprint`, which often contains place information.

In [ ]:
# Read the Excel file directly from the web.
# Pandas will infer the sheet if none is specified.
df = pd.read_excel(EXCEL_URL)

# Standardize column names by stripping accidental whitespace.
df.columns = [c.strip() for c in df.columns]

# Display basic shape and columns for orientation.
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
print("Columns:", list(df.columns))

# Keep only the minimum fields needed for this exercise.
# We create a stable book_id from an existing identifier if available.
preferred_id_cols = ["Identifier", "ID", "id", "Record ID"]
id_col = next((c for c in preferred_id_cols if c in df.columns), None)

if id_col is None:
    # Fall back to row index when no obvious identifier exists.
    df["book_id"] = [f"book_{i}" for i in df.index]
else:
    # Convert to string so graph node IDs remain consistent.
    df["book_id"] = df[id_col].astype(str)

# Ensure Imprint exists and fill missing values for safe NLP processing.
if "Imprint" not in df.columns:
    raise KeyError("Expected an 'Imprint' column, but it was not found in the dataset.")

df["Imprint"] = df["Imprint"].fillna("").astype(str)

df[["book_id", "Imprint"]].head()

## 3) Extract places from `Imprint` using spaCy

To keep the notebook readable, we place the extraction logic in functions.

In [ ]:
# Define helper functions so the extraction logic is reusable and testable.
def normalize_place_name(name: str) -> str:
    """Normalize place strings for deduplication (teaching-friendly simple normalization)."""
    # Remove surrounding spaces and trailing punctuation frequently found in imprints.
    cleaned = name.strip(" .,;:[]()")
    # Use title casing as a simple normalization strategy.
    return cleaned.title()


def extract_places_from_imprint(imprint_text: str, nlp_model) -> List[str]:
    """Extract candidate place names from an imprint string using spaCy GPE/LOC entities."""
    # Parse text with spaCy NER.
    doc = nlp_model(imprint_text)

    # Keep only entities tagged as geopolitical entity or location.
    candidates = [ent.text for ent in doc.ents if ent.label_ in {"GPE", "LOC"}]

    # Normalize and remove empty values.
    normalized = [normalize_place_name(c) for c in candidates if c.strip()]

    # Preserve order while dropping duplicates using dict.fromkeys.
    unique_ordered = list(dict.fromkeys(normalized))
    return unique_ordered


# Apply extraction to every row.
# tqdm shows a live progress bar so long-running pandas operations are easier to monitor.
df["places_extracted"] = df["Imprint"].progress_apply(lambda txt: extract_places_from_imprint(txt, nlp))

# Quick sanity check: how many books have at least one extracted place?
books_with_places = (df["places_extracted"].str.len() > 0).sum()
print(f"Books with >=1 extracted place: {books_with_places:,} / {len(df):,}")

df[["book_id", "Imprint", "places_extracted"]].head(10)

## 4) Disambiguate place names with GeoNames

We query GeoNames and mark names with multiple results as ambiguous.

> GeoNames requires a username. Create one at [geonames.org](https://www.geonames.org/login) and set it as an environment variable before running this section:
>
> ```bash
> export GEONAMES_USERNAME="your_username"
> ```

In [ ]:
# Prepare a long-format table of book-place candidate edges.
# Explode turns one row with a list of places into multiple rows (one place per row).
edges_raw = (
    df[["book_id", "places_extracted"]]
    .explode("places_extracted")
    .rename(columns={"places_extracted": "place_name"})
)

# Drop rows where no place was extracted.
edges_raw = edges_raw.dropna(subset=["place_name"])

# Keep unique candidate edges before geocoding.
edges_raw = edges_raw.drop_duplicates()

print(f"Candidate book-place links: {len(edges_raw):,}")
edges_raw.head()

In [ ]:
# Configure GeoNames client and a throttled geocode function.
# RateLimiter prevents too many rapid API calls.
geonames_username = os.getenv("GEONAMES_USERNAME", "mromanello")
if not geonames_username:
    raise ValueError(
        "Missing GEONAMES_USERNAME. Set it in your environment before running this cell."
    )

geolocator = GeoNames(username=geonames_username, timeout=10)

geo_search = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=0.4,
    max_retries=2,
    error_wait_seconds=1.0,
    swallow_exceptions=False,   # let your try/except handle final failure
)


@lru_cache(maxsize=20_000)
def geonames_candidates(place_name: str, max_rows: int = 5):
    """Return up to max_rows GeoNames candidates for a place name."""
    # geopy returns a list when exactly_one=False.
    results = geo_search(place_name, exactly_one=False)
    return results or []


def disambiguate_place(place_name: str) -> Dict:
    """Select a canonical candidate and mark whether the name is ambiguous."""
    try:
        candidates = geonames_candidates(place_name)

        # If no candidate is found, keep original label and mark unresolved.
        if len(candidates) == 0:
            return {
                "place_name": place_name,
                "place_id": f"place::{place_name}",
                "canonical_name": place_name,
                "latitude": None,
                "longitude": None,
                "country": None,
                "geoname_id": None,
                "ambiguous": False,
                "resolved": False,
                "n_candidates": 0,
            }

        # Choose the first candidate as canonical for this teaching example.
        # More advanced strategies can rank candidates with context.
        top = candidates[0]
        n_candidates = len(candidates)

        return {
            "place_name": place_name,
            "place_id": f"place::{getattr(top, 'geonameId', place_name)}",
            "canonical_name": top.address.split(",")[0] if top.address else place_name,
            "latitude": getattr(top, "latitude", None),
            "longitude": getattr(top, "longitude", None),
            "country": top.raw.get("countryName") if hasattr(top, "raw") else None,
            "geoname_id": getattr(top, "geonameId", None),
            "ambiguous": n_candidates > 1,
            "resolved": True,
            "n_candidates": n_candidates,
        }
    except Exception as exc:
        print(f"GeoNames lookup failed for '{place_name}': {exc}")
        return {
            "place_name": place_name,
            "place_id": f"place::{place_name}",
            "canonical_name": place_name,
            "latitude": None,
            "longitude": None,
            "country": None,
            "geoname_id": None,
            "ambiguous": False,
            "resolved": False,
            "n_candidates": 0,
        }


# Run disambiguation once per unique place to reduce API calls.
unique_places = sorted(edges_raw["place_name"].unique())
print(f"Unique places to disambiguate: {len(unique_places):,}")
print("Disambiguation will be limited to 1000 places due to GeoNames API limits.")

place_records = []
start = time.time()
# GeoNames has a limit of 1000 requests per day, so we limit to 1000 places.
for idx, place in enumerate(unique_places[:1000], start=1):
    # Query GeoNames and append result.
    place_records.append(disambiguate_place(place))

    # Print progress every 100 places for teaching/debugging clarity.
    if idx % 100 == 0:
        elapsed = time.time() - start
        print(f"Processed {idx:,}/{len(unique_places):,} places in {elapsed:.1f}s")

places_df = pd.DataFrame(place_records)
places_df.head()

## 5) Build node and edge tables

We now create clean node tables (`book`, `place`) and an edge table (`book` -> `place`).

In [ ]:
places_df = pd.DataFrame(place_records)
places_df.to_pickle('../../data/disambiguated_place_names.pkl')

In [ ]:
places_df = pd.read_pickle('../../data/disambiguated_place_names.pkl')

In [ ]:
# Merge disambiguated place metadata back into book-place edges.
edges = edges_raw.merge(places_df, on="place_name", how="left")

# Create place node ID fallback for unresolved rows.
edges["place_id"] = edges["place_id"].fillna(edges["place_name"].map(lambda x: f"place::{x}"))

# Build a compact books table with labels.
books_nodes = (
    df[["book_id"]]
    .drop_duplicates()
    .assign(node_type="book", label=lambda d: d["book_id"])
)

# Build place nodes table with disambiguation flags and coordinates.
places_nodes = (
    places_df[["place_id", "canonical_name", "ambiguous", "resolved", "latitude", "longitude", "country"]]
    .drop_duplicates(subset=["place_id"])
    .rename(columns={"canonical_name": "label"})
    .assign(node_type="place")
)

# Keep only columns needed to create graph edges.
edges = edges[["book_id", "place_id", "place_name", "ambiguous", "resolved", "n_candidates"]].drop_duplicates()

print(f"Book nodes:  {len(books_nodes):,}")
print(f"Place nodes: {len(places_nodes):,}")
print(f"Edges:       {len(edges):,}")

edges.head()

## 6) Create bipartite graph in NetworkX

To keep graph construction maintainable, we encapsulate it in a function.

In [ ]:
# Build the bipartite network from tabular nodes/edges.
def build_bipartite_graph(books_df: pd.DataFrame, places_df: pd.DataFrame, edges_df: pd.DataFrame) -> nx.Graph:
    """Create an undirected bipartite graph with book and place node sets."""
    G = nx.Graph()

    # Add book nodes with node_type='book'.
    for _, row in books_df.iterrows():
        G.add_node(
            row["book_id"],
            node_type="book",
            bipartite=0,
            label=row["label"],
        )

    # Add place nodes with disambiguation metadata.
    for _, row in places_df.iterrows():
        G.add_node(
            row["place_id"],
            node_type="place",
            bipartite=1,
            label=row["label"],
            ambiguous=bool(row["ambiguous"]),
            resolved=bool(row["resolved"]),
            latitude=row["latitude"],
            longitude=row["longitude"],
            country=row["country"],
        )

    # Add edges from books to places.
    for _, row in edges_df.iterrows():
        G.add_edge(
            row["book_id"],
            row["place_id"],
            place_name_original=row["place_name"],
            ambiguous=bool(row["ambiguous"]),
            resolved=bool(row["resolved"]),
            n_candidates=int(row["n_candidates"]) if pd.notna(row["n_candidates"]) else 0,
        )

    return G


G = build_bipartite_graph(books_nodes, places_nodes, edges)
print(G)

## 7) Visualize network with ipysigma

We map node color and size to node type and degree. Ambiguous places are highlighted via border color.

In [ ]:
# Precompute degree for visual encoding.
degree_dict = dict(G.degree())
nx.set_node_attributes(G, degree_dict, "degree")

# Keep only place nodes above this degree threshold in the Sigma view.
# All linked book nodes are included automatically.
MIN_PLACE_DEGREE_TO_PLOT = 100

place_nodes_to_keep = {
    n
    for n, attrs in G.nodes(data=True)
    if attrs.get("node_type") == "place" and degree_dict.get(n, 0) > MIN_PLACE_DEGREE_TO_PLOT
}

# Include book neighbors connected to the retained place nodes.
book_neighbors_to_keep = set()
for place_node in place_nodes_to_keep:
    for neighbor in G.neighbors(place_node):
        if G.nodes[neighbor].get("node_type") == "book":
            book_neighbors_to_keep.add(neighbor)

nodes_to_plot = place_nodes_to_keep | book_neighbors_to_keep
G_plot = G.subgraph(nodes_to_plot).copy()

# Recompute degree on the filtered graph so node size matches what is displayed.
plot_degree_dict = dict(G_plot.degree())
nx.set_node_attributes(G_plot, plot_degree_dict, "degree")

# Add a visual category for easier styling in Sigma.
for node, attrs in G_plot.nodes(data=True):
    if attrs.get("node_type") == "book":
        attrs["viz_category"] = "book"
        attrs["border_color"] = "#cccccc"
    else:
        attrs["viz_category"] = "place_ambiguous" if attrs.get("ambiguous") else "place"
        # Highlight ambiguous place nodes with red border.
        attrs["border_color"] = "#d62728" if attrs.get("ambiguous") else "#1f77b4"

n_places = sum(1 for _, attrs in G_plot.nodes(data=True) if attrs.get("node_type") == "place")
n_books = sum(1 for _, attrs in G_plot.nodes(data=True) if attrs.get("node_type") == "book")
print(
    f"Plotting {n_places:,} place nodes (degree > {MIN_PLACE_DEGREE_TO_PLOT}) and "
    f"{n_books:,} linked book nodes; total edges: {G_plot.number_of_edges():,}."
)

# Render interactive graph in the notebook.
Sigma(
    G_plot,
    node_color="viz_category",
    node_size="degree",
    node_label="label",
    node_border_color="border_color",
    default_edge_type="line",
    start_layout=True,
    clickable_edges=False,
    hide_info_panel=False,
)

## 8) Export to GEXF for Gephi Web

Finally, serialize the graph for external exploration tools.

In [ ]:
# Write network to GEXF so it can be loaded in Gephi / Gephi Web.
# Note: NetworkX writes node and edge attributes as part of the GEXF file.
nx.write_gexf(G, OUTPUT_GEXF_PATH)

print(f"GEXF exported to: {OUTPUT_GEXF_PATH}")

# Optional: preview top ambiguous places to discuss uncertainty in class.
ambiguous_places_preview = places_nodes[places_nodes["ambiguous"] == True].head(20)
ambiguous_places_preview